# Stability phase diagram for SGD, HB, and Nesterov

This notebook uses the same synthetic diagonal linear-regression setting as
`D_scaling.ipynb`, but runs a stability scan instead of a scaling-law experiment.

Experiment:

- Fix a very small momentum damping for momentum methods
  \[
  1-\rho.
  \]
- Sweep increasing batch sizes `B`.
- For each `B`, sweep learning rates on a log grid.
- Run **SGD**, **Heavy Ball**, and **Nesterov**.
- Declare a run stable if the final moving average of exact population excess loss
  is below the configured threshold.
- Plot stability boundaries on log-log axes with `B` on the x-axis and `eta` on the
  y-axis.

Expected momentum boundaries:

\[
\eta_{\mathrm{HB}} \sim B(1-\rho),\qquad
\eta_{\mathrm{NAG}} \sim B^\beta(1-\rho).
\]

SGD is included as an empirical stability boundary only; no SGD theory line is drawn.


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------- copied setting from D_scaling.ipynb -----------------------------
BETA  = 1.5
SIGMA = 0
DIM   = 4096
S_VALUE = 3

# ----------------------------- stability experiment parameters -----------------------------
ONE_MINUS_RHO = 2 ** -20
RHO = 1.0 - ONE_MINUS_RHO

K_STEPS = 2000
WINDOW = 50
STABLE_THRESHOLD = 1
DIVERGENCE_LOSS = 1e12

# CPU-friendly default. Increase the upper exponent if you need a wider B range.
B_EXPONENTS = list(range(0, 8))
B_LIST = np.array([2 ** e for e in B_EXPONENTS], dtype=int)

ETA_MIN = 2 ** -10
ETA_MAX = 8
ETA_POINTS = 50
ETA_GRID = np.logspace(np.log10(ETA_MIN), np.log10(ETA_MAX), ETA_POINTS)

ALGOS = {
    'SGD': 'sgd',
    'HB': 'hb',
    'Nesterov': 'nag',
}
ALGO_SEED_OFFSET = {name: k for k, name in enumerate(ALGOS)}

SEED = 0
CHUNK = 2048
LOAD_CACHE = True
SAVE_CACHE = True
RESULT_DIR = Path('results')
RESULT_DIR.mkdir(exist_ok=True)

print(f'beta={BETA:g}, sigma={SIGMA:g}, dim={DIM}, s={S_VALUE:g}')
print(f'1-rho={ONE_MINUS_RHO:.2e}, rho={RHO:.12f} (momentum methods only)')
print(f'B grid: 2^{B_EXPONENTS[0]} ... 2^{B_EXPONENTS[-1]} ({len(B_LIST)} values)')
print(f'eta grid: {ETA_MIN:.1e} ... {ETA_MAX:.1e} ({len(ETA_GRID)} values)')
print(f'stability: final {WINDOW}-step moving-average loss < {STABLE_THRESHOLD:g} after {K_STEPS} steps')


## Linear-regression problem and stochastic gradient

We work in the eigenbasis of the diagonal covariance matrix.  The exact population
excess loss is

\[
\mathcal E(\theta)=\frac12\sum_j \lambda_j(\theta_j-\theta^*_j)^2.
\]

The stochastic gradient is computed in chunks to avoid materializing very large
`(B, DIM)` matrices when `B` is large.


In [10]:
def build_problem(dim=DIM, beta=BETA, s=S_VALUE):
    """Return (lam, theta_star) for the diagonal linear-regression problem."""
    j = np.arange(1, dim + 1, dtype=np.float64)
    lam = j ** (-beta)
    var_star = j ** (-(1.0 + (s - 1.0) * beta))
    theta_star = np.sqrt(var_star)
    return lam, theta_star


def excess_loss(delta, lam):
    """Exact population excess loss E = 1/2 * sum_j lam_j * delta_j^2."""
    return 0.5 * np.sum(lam * delta * delta)


def grad_delta_chunked(delta, B, lam, sigma=SIGMA, rng=None, chunk=CHUNK):
    """Stochastic minibatch gradient without materializing a full (B, DIM) batch."""
    rng = rng or np.random.default_rng()
    d = lam.shape[0]
    sqrt_lam = np.sqrt(lam)
    grad = np.zeros(d, dtype=np.float64)
    remaining = int(B)
    while remaining > 0:
        b = min(chunk, remaining)
        X = rng.standard_normal((b, d)) * sqrt_lam
        eps = rng.standard_normal(b) * sigma
        resid = X @ delta - eps
        grad += X.T @ resid
        remaining -= b
    return grad / B


lam, theta_star = build_problem()
E0 = excess_loss(-theta_star, lam)
print(f'lambda range [{lam[-1]:.2e}, {lam[0]:.2e}], initial excess loss E(0)={E0:.4f}')


lambda range [3.81e-06, 1.00e+00], initial excess loss E(0)=0.5126


## Stability test

For each `(algorithm, B, eta)`, run exactly `K_STEPS=10000` updates.  The final
100-step moving average is

\[
\frac1{100}\sum_{t=K-99}^{K} \mathcal E_t.
\]

The run is marked stable if this value is finite and below `STABLE_THRESHOLD`.
For each batch size `B`, the grid scan also records the lowest final moving-average
loss over all tested learning rates, together with the learning rate that achieved it.
Very large losses or non-finite values are treated as unstable and stopped early.


In [ ]:
def run_stability(algo, B, eta, rho=RHO, K=K_STEPS, window=WINDOW,
                  threshold=STABLE_THRESHOLD, divergence_loss=DIVERGENCE_LOSS,
                  seed=SEED):
    """Return a dict with stable flag, final moving average, and final loss."""
    rng = np.random.default_rng(seed)
    rho_eff = 0.0 if algo == 'sgd' else rho
    delta = -theta_star.copy()
    delta_prev = delta.copy()
    tail = np.empty(window, dtype=np.float64)
    tail[:] = np.nan

    loss0 = excess_loss(delta, lam)
    tail[0 % window] = loss0

    for t in range(1, K + 1):
        momentum = delta - delta_prev
        eval_delta = delta + rho_eff * momentum if algo == 'nag' else delta
        g = grad_delta_chunked(eval_delta, B, lam, rng=rng)
        delta_new = delta - eta * g + rho_eff * momentum
        delta_prev, delta = delta, delta_new

        loss = excess_loss(delta, lam)
        if (not np.isfinite(loss)) or loss > divergence_loss:
            return {
                'stable': False,
                'final_ma': np.inf,
                'final_loss': float(loss) if np.isfinite(loss) else np.inf,
                'steps': t,
                'reason': 'diverged',
            }
        tail[t % window] = loss

    final_ma = float(np.mean(tail))
    return {
        'stable': bool(np.isfinite(final_ma) and final_ma < threshold),
        'final_ma': final_ma,
        'final_loss': float(loss),
        'steps': K,
        'reason': 'finished',
    }


# quick sanity check with a tiny run; comment out if not needed
quick = run_stability('hb', B=1, eta=ETA_GRID[0], K=10, window=10)
quick


## Run the stability grid

The full default grid is intentionally large: `2 algorithms × len(B_LIST) × len(ETA_GRID)`
runs, each up to `10000` steps.  Results are cached under `results/` so rerunning the
notebook can reuse finished scans.


In [ ]:
def cache_path():
    tag = (
        f'stability_phase_3alg_beta{BETA:g}_s{S_VALUE:g}_dim{DIM}_'
        f'B{B_EXPONENTS[0]}-{B_EXPONENTS[-1]}_eta{ETA_POINTS}_'
        f'omrho{ONE_MINUS_RHO:.0e}_K{K_STEPS}_w{WINDOW}_seed{SEED}'
    ).replace('.', '_').replace('-', 'm')
    return RESULT_DIR / f'{tag}.npz'


def summarize_min_loss_by_B(final_ma, final_loss):
    """For each algorithm and B, record the lowest loss over the eta grid."""
    min_final_ma = {}
    min_final_ma_eta = {}
    min_final_loss = {}
    min_final_loss_eta = {}

    for name in ALGOS:
        min_final_ma[name] = np.full(len(B_LIST), np.nan)
        min_final_ma_eta[name] = np.full(len(B_LIST), np.nan)
        min_final_loss[name] = np.full(len(B_LIST), np.nan)
        min_final_loss_eta[name] = np.full(len(B_LIST), np.nan)

        for j in range(len(B_LIST)):
            ma_col = np.asarray(final_ma[name][:, j], dtype=float)
            loss_col = np.asarray(final_loss[name][:, j], dtype=float)

            ma_finite = np.isfinite(ma_col)
            if np.any(ma_finite):
                idx = int(np.argmin(np.where(ma_finite, ma_col, np.inf)))
                min_final_ma[name][j] = ma_col[idx]
                min_final_ma_eta[name][j] = ETA_GRID[idx]

            loss_finite = np.isfinite(loss_col)
            if np.any(loss_finite):
                idx = int(np.argmin(np.where(loss_finite, loss_col, np.inf)))
                min_final_loss[name][j] = loss_col[idx]
                min_final_loss_eta[name][j] = ETA_GRID[idx]

    return min_final_ma, min_final_ma_eta, min_final_loss, min_final_loss_eta


path = cache_path()
if LOAD_CACHE and path.exists():
    data = np.load(path, allow_pickle=True)
    STABLE = data['stable'].item()
    FINAL_MA = data['final_ma'].item()
    FINAL_LOSS = data['final_loss'].item()
    STEPS_DONE = data['steps_done'].item()
    if 'min_final_ma' in data.files:
        MIN_FINAL_MA = data['min_final_ma'].item()
        MIN_FINAL_MA_ETA = data['min_final_ma_eta'].item()
        MIN_FINAL_LOSS = data['min_final_loss'].item()
        MIN_FINAL_LOSS_ETA = data['min_final_loss_eta'].item()
    else:
        MIN_FINAL_MA, MIN_FINAL_MA_ETA, MIN_FINAL_LOSS, MIN_FINAL_LOSS_ETA = summarize_min_loss_by_B(FINAL_MA, FINAL_LOSS)
    print(f'loaded cached stability grid from {path}')
else:
    STABLE = {name: np.zeros((len(ETA_GRID), len(B_LIST)), dtype=bool) for name in ALGOS}
    FINAL_MA = {name: np.full((len(ETA_GRID), len(B_LIST)), np.nan) for name in ALGOS}
    FINAL_LOSS = {name: np.full((len(ETA_GRID), len(B_LIST)), np.nan) for name in ALGOS}
    STEPS_DONE = {name: np.zeros((len(ETA_GRID), len(B_LIST)), dtype=int) for name in ALGOS}

    for name, algo in ALGOS.items():
        print(f'=== {name} ===')
        for j, B in enumerate(B_LIST):
            print(f'B=2^{int(np.log2(B)):<2d} ({B:6d})')
            for i, eta in enumerate(ETA_GRID):
                seed = SEED + 1_000_000 * ALGO_SEED_OFFSET[name] + 10_000 * j + i
                out = run_stability(algo, int(B), float(eta), seed=seed)
                STABLE[name][i, j] = out['stable']
                FINAL_MA[name][i, j] = out['final_ma']
                FINAL_LOSS[name][i, j] = out['final_loss']
                STEPS_DONE[name][i, j] = out['steps']

            n_stable = int(STABLE[name][:, j].sum())
            finite = np.isfinite(FINAL_MA[name][:, j])
            if np.any(finite):
                best_i = int(np.argmin(np.where(finite, FINAL_MA[name][:, j], np.inf)))
                print(f'  stable eta count: {n_stable}/{len(ETA_GRID)}; '
                      f'min final-MA loss={FINAL_MA[name][best_i, j]:.3e} at eta={ETA_GRID[best_i]:.3e}')
            else:
                print(f'  stable eta count: {n_stable}/{len(ETA_GRID)}; min final-MA loss=nan')

    MIN_FINAL_MA, MIN_FINAL_MA_ETA, MIN_FINAL_LOSS, MIN_FINAL_LOSS_ETA = summarize_min_loss_by_B(FINAL_MA, FINAL_LOSS)

    if SAVE_CACHE:
        np.savez_compressed(
            path,
            stable=STABLE,
            final_ma=FINAL_MA,
            final_loss=FINAL_LOSS,
            steps_done=STEPS_DONE,
            min_final_ma=MIN_FINAL_MA,
            min_final_ma_eta=MIN_FINAL_MA_ETA,
            min_final_loss=MIN_FINAL_LOSS,
            min_final_loss_eta=MIN_FINAL_LOSS_ETA,
            B_LIST=B_LIST,
            ETA_GRID=ETA_GRID,
        )
        print(f'saved stability grid to {path}')


## Plot stability regions in one log-log figure and save PDF

This plot puts SGD, HB, and Nesterov in the same axes. Points are empirical largest
stable learning rates at each batch size. Dashed lines are fitted references for the
momentum methods only:

\[
\eta_{\mathrm{HB}} \propto B(1-\rho),\qquad
\eta_{\mathrm{NAG}} \propto B^\beta(1-\rho).
\]

SGD is shown as an empirical green boundary with no theory line. The background has
three colored regions based on the blue HB and orange Nesterov boundaries: below HB,
between HB and Nesterov, and above Nesterov. The x-axis starts at $B=2^0$ when
`B_EXPONENTS[0] = 0`.


In [ ]:
def max_stable_eta(stable_map):
    """Largest stable eta for each B; NaN if no eta is stable."""
    boundary = np.full(len(B_LIST), np.nan)
    for j in range(len(B_LIST)):
        idx = np.where(stable_map[:, j])[0]
        if len(idx):
            boundary[j] = ETA_GRID[idx.max()]
    return boundary


def fit_prefactor(boundary, power):
    """Fit C in eta ~= C * B^power * (1-rho) using finite boundary points."""
    mask = np.isfinite(boundary) & (boundary > 0)
    if not np.any(mask):
        return 1.0
    denom = (B_LIST[mask].astype(float) ** power) * ONE_MINUS_RHO
    return float(np.exp(np.mean(np.log(boundary[mask] / denom))))


def stability_pdf_path():
    tag = (
        f'stability_regions_3alg_beta{BETA:g}_s{S_VALUE:g}_dim{DIM}_'
        f'B{B_EXPONENTS[0]}-{B_EXPONENTS[-1]}_eta{ETA_POINTS}_'
        f'omrho{ONE_MINUS_RHO:.0e}_K{K_STEPS}_w{WINDOW}_thr{STABLE_THRESHOLD:g}'
    ).replace('.', '_').replace('-', 'm')
    return RESULT_DIR / f'{tag}.pdf'


fig, ax = plt.subplots(figsize=(7.8, 5.8))

styles = {
    'SGD': {
        'color': 'tab:green',
        'marker': '^',
        'power': None,
        'theory_label': None,
    },
    'HB': {
        'color': 'tab:blue',
        'marker': 'o',
        'power': 1.0,
        'theory_label': r'HB fit: $\eta \propto B(1-\rho)$',
    },
    'Nesterov': {
        'color': 'tab:orange',
        'marker': 's',
        'power': BETA,
        'theory_label': rf'Nesterov fit: $\eta \propto B^{{\beta}}(1-\rho)$, $\beta={BETA:g}$',
    },
}

BOUNDARIES = {}
REFS = {}
for name in ALGOS:
    boundary = max_stable_eta(STABLE[name])
    BOUNDARIES[name] = boundary
    style = styles[name]
    if style['power'] is not None:
        C = fit_prefactor(boundary, style['power'])
        REFS[name] = C * (B_LIST.astype(float) ** style['power']) * ONE_MINUS_RHO

eta_low = float(np.min(ETA_GRID))
eta_high = float(np.max(ETA_GRID))
hb_region = np.where(np.isfinite(BOUNDARIES['HB']), BOUNDARIES['HB'], REFS['HB'])
nes_region = np.where(np.isfinite(BOUNDARIES['Nesterov']), BOUNDARIES['Nesterov'], REFS['Nesterov'])
hb_region = np.clip(hb_region, eta_low, eta_high)
nes_region = np.clip(nes_region, eta_low, eta_high)

# Three requested regions: below blue, between blue/orange, above orange.
ax.fill_between(
    B_LIST,
    eta_low,
    hb_region,
    color='#c6dbef',
    alpha=0.55,
    label='below HB boundary (blue)',
    zorder=0,
)
ax.fill_between(
    B_LIST,
    hb_region,
    nes_region,
    color='#ddd6f3',
    alpha=0.60,
    label='between HB and Nesterov boundaries',
    zorder=0,
)
ax.fill_between(
    B_LIST,
    nes_region,
    eta_high,
    color='#fdd0a2',
    alpha=0.50,
    label='above Nesterov boundary (orange)',
    zorder=0,
)

for name in ALGOS:
    style = styles[name]
    boundary = BOUNDARIES[name]
    mask = np.isfinite(boundary) & (boundary > 0)
    ax.plot(
        B_LIST[mask], boundary[mask],
        color=style['color'], marker=style['marker'], linewidth=2.4,
        label=rf'{name}: empirical max stable $\eta$',
        zorder=3,
    )
    if style['theory_label'] is not None:
        ax.plot(
            B_LIST, REFS[name], linestyle='--', color=style['color'], alpha=0.9,
            linewidth=2.0, label=style['theory_label'], zorder=2,
        )

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlim(2 ** B_EXPONENTS[0], B_LIST[-1])
ax.set_ylim(eta_low, eta_high)
ax.set_xlabel('batch size  $B$')
ax.set_ylabel(r'learning rate stability boundary  $\eta$')
ax.set_title('SGD, HB, and Nesterov stability regions')
ax.grid(True, which='both', alpha=0.3)

legend_title = (
    rf'$1-\rho={ONE_MINUS_RHO:.0e}$ for HB/NAG, $\beta={BETA:g}$, $s={S_VALUE:g}$' + '\n' +
    rf'$K={K_STEPS}$, MA window={WINDOW}, threshold={STABLE_THRESHOLD:g}' + '\n' +
    rf'$B=2^{{{B_EXPONENTS[0]}}}\ldots2^{{{B_EXPONENTS[-1]}}}$, '
    rf'$\eta\in[{ETA_MIN:.0e},{ETA_MAX:.0e}]$'
)
ax.legend(title=legend_title, fontsize=7.5, title_fontsize=8, loc='best')

fig.tight_layout()
PDF_PATH = stability_pdf_path()
fig.savefig(PDF_PATH, bbox_inches='tight')
print(f'saved PDF to {PDF_PATH}')
plt.show()


## Boundary and minimum-loss table

This table prints both the largest stable `eta` and the lowest final moving-average
loss for each batch size.  The minimum-loss column is useful when no tested learning
rate satisfies the stability threshold, or when comparing how close each `B` gets to
stable behavior.


In [ ]:
for name in ALGOS:
    boundary = max_stable_eta(STABLE[name])
    print(f'\n{name}: max stable eta and minimum loss by B')
    for j, (B, eta_max) in enumerate(zip(B_LIST, boundary)):
        eta_text = f'{eta_max:.3e}' if np.isfinite(eta_max) else 'nan'
        min_ma = MIN_FINAL_MA[name][j]
        min_ma_eta = MIN_FINAL_MA_ETA[name][j]
        min_loss = MIN_FINAL_LOSS[name][j]
        min_loss_eta = MIN_FINAL_LOSS_ETA[name][j]
        min_ma_text = f'{min_ma:.3e} @ eta={min_ma_eta:.3e}' if np.isfinite(min_ma) else 'nan'
        min_loss_text = f'{min_loss:.3e} @ eta={min_loss_eta:.3e}' if np.isfinite(min_loss) else 'nan'
        print(f'  B={B:6d}  eta_max={eta_text:>9s}  min_final_MA={min_ma_text}  min_final_loss={min_loss_text}')

    mask = np.isfinite(boundary) & (boundary > 0)
    if mask.sum() >= 2:
        slope, intercept = np.polyfit(np.log(B_LIST[mask]), np.log(boundary[mask]), 1)
        print(f'  fitted boundary slope d log(eta) / d log(B) = {slope:.3f}')
